In [1]:
# ============================================================
# Load the cleaned UPI dataset and create the SQLite database
# ============================================================

import sqlite3
from pathlib import Path
import pandas as pd


# Load the cleaned dataset
df = pd.read_csv(
    "../data/clean/upi_clean.csv",
    parse_dates=["date"]
)


# Convert dates into SQLite-compatible text format
df["date"] = df["date"].dt.strftime("%Y-%m-%d")


# Connect to the SQLite database
con = sqlite3.connect("../data/upi.db")


# Create the database table using the SQL schema
con.executescript(
    Path("../sql/schema.sql").read_text()
)


# Insert the cleaned dataset into SQLite
df.to_sql(
    "upi_monthly",
    con,
    if_exists="append",
    index=False
)


# Verify the number of rows loaded
print(
    pd.read_sql_query(
        "SELECT COUNT(*) AS rows FROM upi_monthly",
        con
    )
)

   rows
0    66


In [2]:
# ============================================================
# Verify the SQLite table structure
# ============================================================

query = """
PRAGMA table_info(upi_monthly);
"""

table_info = pd.read_sql_query(query, con)

print(table_info[["name", "type"]])

                    name     type
0                   date     TEXT
1                  month     TEXT
2         financial_year     TEXT
3           month_number  INTEGER
4             month_name     TEXT
5             banks_live  INTEGER
6              volume_mn     REAL
7               value_cr     REAL
8        avg_ticket_size     REAL
9          days_in_month  INTEGER
10   avg_daily_volume_mn     REAL
11    avg_daily_value_cr     REAL
12        mom_growth_pct     REAL
13        yoy_growth_pct     REAL
14  value_mom_growth_pct     REAL
15  value_yoy_growth_pct     REAL
16   fy_months_available  INTEGER
17           source_file     TEXT


In [4]:
# ============================================================
# Create a helper function to run named SQL queries
# ============================================================

def run(name):
    sql_text = Path("../sql/queries.sql").read_text()

    query = (
        sql_text
        .split(f"-- name: {name}")[1]
        .split("-- name:")[0]
    )

    return pd.read_sql_query(query, con)

In [5]:
# ============================================================
# Run the financial-year summary SQL query
# ============================================================

fy_summary = run("fy_summary")

print(fy_summary)


  financial_year  months  volume_bn  value_lakh_cr  avg_ticket_rs
0        2021-22      12       46.0           84.2         1831.0
1        2022-23      12       83.8          139.2         1662.0
2        2023-24      12      131.1          200.0         1525.0
3        2024-25      12      185.9          260.6         1402.0
4        2025-26      12      241.6          314.2         1301.0
5        2026-27       6      140.5          176.9         1259.0


In [6]:
# ============================================================
# Compare April-September UPI performance across financial years
# ============================================================

h1_summary = run("like_for_like_h1")

print(h1_summary)

  financial_year  volume_bn  volume_yoy_pct  value_yoy_pct
0        2021-22       18.4             NaN            NaN
1        2022-23       37.0           100.9           83.3
2        2023-24       58.7            58.5           44.2
3        2024-25       85.7            45.9           34.6
4        2025-26      114.1            33.2           21.2
5        2026-27      140.5            23.2           19.6


In [7]:
# ============================================================
# Find the top 5 months with the strongest daily volume growth
# ============================================================

top_growth = run("top_growth_months")

print(top_growth)

           month  daily_mom_pct  rnk
0      June-2021           14.2    1
1      July-2021           12.0    2
2   October-2021           11.7    3
3    August-2021            9.5    4
4  February-2022            8.6    5


In [8]:
# ============================================================
# Compare festive months with other months
# ============================================================

festive_summary = run("festive_vs_other")

print(festive_summary)

              period  months  avg_daily_mom_pct
0  Festive (Oct-Nov)      10               3.56
1       Other months      55               3.50


In [9]:
# ============================================================
# Final validation of the SQLite database
# ============================================================

validation = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total_rows,
        MIN(date) AS first_date,
        MAX(date) AS last_date,
        COUNT(DISTINCT date) AS unique_dates
    FROM upi_monthly;
    """,
    con
)

print(validation)

   total_rows  first_date   last_date  unique_dates
0          66  2021-04-01  2026-09-01            66


In [10]:
# ============================================================
# Close the SQLite database connection safely
# ============================================================

con.close()

print("SQLite database connection closed successfully.")

SQLite database connection closed successfully.
